# PolaFusion — Model Training

Cell 1: 5-fold mDeBERTa-v3-base training for Subtasks 1, 2, 3  
Cell 2: 3-fold XLM-RoBERTa-large training for Subtasks 1, 2, 3  

**Run order:** Run each cell independently on Kaggle with GPU

In [ ]:
# 5-fold mDeBERTa training for Subtasks 1, 2, 3
import os
import gc
import shutil
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.model_selection import KFold, StratifiedKFold
from sklearn.metrics import f1_score, accuracy_score
from torch.utils.data import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from transformers import DataCollatorWithPadding

MODEL_NAME = "microsoft/mdeberta-v3-base"
N_FOLDS = 5
MAX_LEN = 128
BATCH_SIZE = 16
LEARNING_RATE = 2e-5
data_dir = "/kaggle/input/semeval-combined-dataset-evaluation-phase/SEMEVAL"

TASK_CONFIGS = {
    1: {
        "target_cols": ["polarization"],
        "epochs": 3,
        "num_labels": 1,
        "problem_type": "regression",
        "metric_for_best_model": "eval_loss",
        "greater_is_better": False,
        "stratify": False,
    },
    2: {
        "target_cols": ["political", "racial/ethnic", "religious", "gender/sexual", "other"],
        "epochs": 4,
        "num_labels": 5,
        "problem_type": "multi_label_classification",
        "metric_for_best_model": "f1_macro",
        "greater_is_better": True,
        "stratify": True,
    },
    3: {
        "target_cols": ["stereotype", "vilification", "dehumanization", "extreme_language", "lack_of_empathy", "invalidation"],
        "epochs": 4,
        "num_labels": 6,
        "problem_type": "multi_label_classification",
        "metric_for_best_model": "f1_macro",
        "greater_is_better": True,
        "stratify": True,
    },
}

# --- CUSTOM LOSS ---
class FocalLoss(nn.Module):
    def __init__(self, alpha=1, gamma=2, reduction='mean'):
        super(FocalLoss, self).__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction
        self.bce = nn.BCEWithLogitsLoss(reduction='none')

    def forward(self, inputs, targets):
        bce_loss = self.bce(inputs, targets)
        pt = torch.exp(-bce_loss)
        focal_loss = self.alpha * (1 - pt) ** self.gamma * bce_loss
        return focal_loss.mean() if self.reduction == 'mean' else focal_loss.sum()

class FocalTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")
        loss_fct = FocalLoss(gamma=2.0)
        loss = loss_fct(logits, labels)
        return (loss, outputs) if return_outputs else loss

# --- DATASET ---
class TaskDataset(Dataset):
    def __init__(self, df, tokenizer, target_cols):
        self.texts = df['text'].astype(str).values
        self.labels = df[target_cols].values
        self.tokenizer = tokenizer
    def __len__(self): return len(self.texts)
    def __getitem__(self, idx):
        enc = self.tokenizer(self.texts[idx], truncation=True, padding="max_length", max_length=MAX_LEN)
        item = {key: torch.tensor(val) for key, val in enc.items()}
        item['labels'] = torch.tensor(self.labels[idx], dtype=torch.float)
        return item

# --- METRICS ---
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs = 1 / (1 + np.exp(-logits))
    predictions = (probs > 0.5).astype(int)
    f1 = f1_score(labels, predictions, average='macro')
    acc = accuracy_score(labels, predictions)
    return {"f1_macro": f1, "accuracy": acc}

# --- MAIN LOOP OVER ALL SUBTASKS ---
for TASK_ID, cfg in TASK_CONFIGS.items():
    print(f"\n{'='*60}")
    print(f"🚀 Starting 5-Fold mDeBERTa Training for Subtask {TASK_ID}...")
    print(f"{'='*60}")

    TARGET_COLS = cfg["target_cols"]
    OUTPUT_DIR = f"./subtask{TASK_ID}_5fold_output"
    FINAL_MODELS_DIR = f"./subtask{TASK_ID}_final_models"

    df = pd.read_csv(f"{data_dir}/subtask{TASK_ID}_full_train.csv")
    df = df.dropna(subset=TARGET_COLS)

    if TASK_ID == 1:
        df['polarization'] = df['polarization'].astype(int)
    else:
        df['sum'] = df[TARGET_COLS].sum(axis=1)
        df = df[df['sum'] > 0].reset_index(drop=True)
        df['stratify_key'] = df['language'] + "_" + df['sum'].astype(str)

    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

    if cfg["stratify"]:
        skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)
        fold_iter = skf.split(df, df['stratify_key'])
    else:
        kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=42)
        fold_iter = kf.split(df)

    for fold, (train_idx, val_idx) in enumerate(fold_iter):
        print(f"\nTraining Fold {fold + 1}/{N_FOLDS}")

        if fold > 0:
            prev_dir = f"{OUTPUT_DIR}/fold_{fold-1}"
            if os.path.exists(prev_dir):
                shutil.rmtree(prev_dir)

        train_df = df.iloc[train_idx]
        val_df = df.iloc[val_idx]

        train_ds = TaskDataset(train_df, tokenizer, TARGET_COLS)
        val_ds = TaskDataset(val_df, tokenizer, TARGET_COLS)

        model = AutoModelForSequenceClassification.from_pretrained(
            MODEL_NAME, num_labels=cfg["num_labels"], problem_type=cfg["problem_type"]
        )

        fold_checkpoint_dir = f"{OUTPUT_DIR}/fold_{fold}"

        args = TrainingArguments(
            output_dir=fold_checkpoint_dir,
            learning_rate=LEARNING_RATE,
            per_device_train_batch_size=BATCH_SIZE,
            per_device_eval_batch_size=BATCH_SIZE * 2,
            num_train_epochs=cfg["epochs"],
            weight_decay=0.01,
            eval_strategy="epoch",
            save_strategy="epoch",
            save_total_limit=1,
            load_best_model_at_end=True,
            metric_for_best_model=cfg["metric_for_best_model"],
            greater_is_better=cfg["greater_is_better"],
            fp16=True,
            report_to="none"
        )

        trainer = FocalTrainer(
            model=model,
            args=args,
            train_dataset=train_ds,
            eval_dataset=val_ds,
            processing_class=tokenizer,
            data_collator=data_collator,
            compute_metrics=compute_metrics
        )

        trainer.train()

        final_model_path = f"{FINAL_MODELS_DIR}/fold_{fold}"
        trainer.save_model(final_model_path)
        print(f"   ✅ Saved model to: {final_model_path}")

        if os.path.exists(fold_checkpoint_dir):
            shutil.rmtree(fold_checkpoint_dir)
            print(f"   🗑️ DELETED checkpoints: {fold_checkpoint_dir}")

        del model, trainer
        torch.cuda.empty_cache()
        gc.collect()

    print(f"\n✅ Subtask {TASK_ID} 5-Fold Training Complete! Models in {FINAL_MODELS_DIR}")
    del tokenizer
    gc.collect()

In [ ]:
# 3-fold XLM-RoBERTa-large training for Subtasks 1, 2, 3
import os
import gc
import shutil
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import f1_score, accuracy_score
from torch.utils.data import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from transformers import DataCollatorWithPadding

MODEL_NAME = "xlm-roberta-large"
N_FOLDS = 3
MAX_LEN = 128
BATCH_SIZE = 8
GRADIENT_ACCUMULATION = 2
LEARNING_RATE = 1e-5
data_dir = "/kaggle/input/semeval-combined-dataset-evaluation-phase/SEMEVAL"

TASK_CONFIGS = {
    1: {
        "target_cols": ["polarization"],
        "epochs": 3,
        "num_labels": 1,
        "problem_type": "regression",
        "save_strategy": "no",
        "load_best_model_at_end": False,
        "metric_for_best_model": None,
        "greater_is_better": None,
        "filter_polarized": False,
    },
    2: {
        "target_cols": ["political", "racial/ethnic", "religious", "gender/sexual", "other"],
        "epochs": 4,
        "num_labels": 5,
        "problem_type": "multi_label_classification",
        "save_strategy": "epoch",
        "load_best_model_at_end": True,
        "metric_for_best_model": "f1_macro",
        "greater_is_better": True,
        "filter_polarized": True,
    },
    3: {
        "target_cols": ["stereotype", "vilification", "dehumanization", "extreme_language", "lack_of_empathy", "invalidation"],
        "epochs": 4,
        "num_labels": 6,
        "problem_type": "multi_label_classification",
        "save_strategy": "epoch",
        "load_best_model_at_end": True,
        "metric_for_best_model": "f1_macro",
        "greater_is_better": True,
        "filter_polarized": True,
    },
}

# --- CUSTOM LOSS ---
class FocalLoss(nn.Module):
    def __init__(self, alpha=1, gamma=2, reduction='mean'):
        super(FocalLoss, self).__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction
        self.bce = nn.BCEWithLogitsLoss(reduction='none')

    def forward(self, inputs, targets):
        bce_loss = self.bce(inputs, targets)
        pt = torch.exp(-bce_loss)
        focal_loss = self.alpha * (1 - pt) ** self.gamma * bce_loss
        return focal_loss.mean() if self.reduction == 'mean' else focal_loss.sum()

class FocalTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")
        loss_fct = FocalLoss(gamma=2.0)
        loss = loss_fct(logits, labels)
        return (loss, outputs) if return_outputs else loss

# --- DATASET ---
class TaskDataset(Dataset):
    def __init__(self, df, tokenizer, target_cols):
        self.texts = df['text'].astype(str).values
        self.labels = df[target_cols].values
        self.tokenizer = tokenizer
    def __len__(self): return len(self.texts)
    def __getitem__(self, idx):
        enc = self.tokenizer(self.texts[idx], truncation=True, padding="max_length", max_length=MAX_LEN)
        item = {key: torch.tensor(val) for key, val in enc.items()}
        item['labels'] = torch.tensor(self.labels[idx], dtype=torch.float)
        return item

# --- METRICS ---
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs = 1 / (1 + np.exp(-logits))
    predictions = (probs > 0.5).astype(int)
    f1 = f1_score(labels, predictions, average='macro')
    acc = accuracy_score(labels, predictions)
    return {"f1_macro": f1, "accuracy": acc}

# --- MAIN LOOP OVER ALL SUBTASKS ---
for TASK_ID, cfg in TASK_CONFIGS.items():
    print(f"\n{'='*60}")
    print(f"🚀 Starting 3-Fold XLM-R Large Training for Subtask {TASK_ID}...")
    print(f"{'='*60}")

    TARGET_COLS = cfg["target_cols"]
    OUTPUT_DIR = f"./subtask{TASK_ID}_xlmr_large_3fold_output"
    FINAL_MODELS_DIR = f"./subtask{TASK_ID}_xlmr_large_final_models"

    # Clean workspace
    for d in [OUTPUT_DIR, FINAL_MODELS_DIR]:
        if os.path.exists(d):
            shutil.rmtree(d)
    print("🧹 Workspace wiped clean.")

    df = pd.read_csv(f"{data_dir}/subtask{TASK_ID}_full_train.csv")
    df = df.dropna(subset=TARGET_COLS)

    if TASK_ID == 1:
        df['polarization'] = df['polarization'].astype(int)
        df['stratify_key'] = df['language'] + "_" + df['polarization'].astype(str)
    else:
        df['sum'] = df[TARGET_COLS].sum(axis=1)
        df = df[df['sum'] > 0].reset_index(drop=True)
        df['label_str'] = df[TARGET_COLS].astype(int).astype(str).agg('_'.join, axis=1)
        df['stratify_key'] = df['language'] + "_" + df['sum'].astype(str)

    le = LabelEncoder()
    df['stratify_label'] = le.fit_transform(df['stratify_key'])

    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)

    for fold, (train_idx, val_idx) in enumerate(skf.split(df, df['stratify_label'])):
        print(f"\nTraining Fold {fold + 1}/{N_FOLDS}")

        if fold > 0:
            prev_dir = f"{OUTPUT_DIR}/fold_{fold-1}"
            if os.path.exists(prev_dir):
                shutil.rmtree(prev_dir)
                print(f"   🧹 Cleaned up: {prev_dir}")

        train_df = df.iloc[train_idx]
        val_df = df.iloc[val_idx]

        train_ds = TaskDataset(train_df, tokenizer, TARGET_COLS)
        val_ds = TaskDataset(val_df, tokenizer, TARGET_COLS)

        model = AutoModelForSequenceClassification.from_pretrained(
            MODEL_NAME, num_labels=cfg["num_labels"], problem_type=cfg["problem_type"]
        )

        fold_checkpoint_dir = f"{OUTPUT_DIR}/fold_{fold}"

        training_args_kwargs = dict(
            output_dir=fold_checkpoint_dir,
            learning_rate=LEARNING_RATE,
            per_device_train_batch_size=BATCH_SIZE,
            gradient_accumulation_steps=GRADIENT_ACCUMULATION,
            per_device_eval_batch_size=BATCH_SIZE * 2,
            num_train_epochs=cfg["epochs"],
            weight_decay=0.01,
            eval_strategy="epoch",
            save_strategy=cfg["save_strategy"],
            load_best_model_at_end=cfg["load_best_model_at_end"],
            fp16=True,
            report_to="none"
        )
        if cfg["metric_for_best_model"]:
            training_args_kwargs["metric_for_best_model"] = cfg["metric_for_best_model"]
            training_args_kwargs["greater_is_better"] = cfg["greater_is_better"]
            training_args_kwargs["save_total_limit"] = 1
            training_args_kwargs["save_only_model"] = True

        args = TrainingArguments(**training_args_kwargs)

        trainer = FocalTrainer(
            model=model,
            args=args,
            train_dataset=train_ds,
            eval_dataset=val_ds,
            tokenizer=tokenizer,
            data_collator=data_collator,
            compute_metrics=compute_metrics
        )

        trainer.train()

        final_path = f"{FINAL_MODELS_DIR}/fold_{fold}"
        trainer.save_model(final_path)
        print(f"   ✅ Saved model to: {final_path}")

        if os.path.exists(fold_checkpoint_dir):
            try:
                shutil.rmtree(fold_checkpoint_dir)
                print(f"   🗑️ DELETED Temp Checkpoints: {fold_checkpoint_dir}")
            except Exception as e:
                print(f"   ⚠️ Cleanup Warning: {e}")

        del model, trainer
        torch.cuda.empty_cache()
        gc.collect()

    print(f"\n✅ Subtask {TASK_ID} 3-Fold XLM-R Training Complete! Models in {FINAL_MODELS_DIR}")
    del tokenizer
    gc.collect()